In [ ]:
from pathlib import Path

from atlas.common.config.loader import get_settings
from atlas.common.spark.session import get_spark_session

project_root = Path.cwd()

while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
settings = get_settings(
    project_root / "configs" / "base.yaml",
    project_root / "configs" / "local.yaml",
    project_root / "pyproject.toml",
)


In [ ]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

In [ ]:
customer_cdc_stream = (spark.readStream.format("kafka")
                       .option("kafka.bootstrap.servers", settings.kafka.bootstrap_servers)
                       .option("subscribe", "atlas.customer.public.customers")
                       .option("startingOffsets", "earliest")
                       .load()
                       )

In [ ]:
customer_cdc_stream.printSchema()

In [ ]:
customer_converted = customer_cdc_stream.selectExpr("CAST(key as String) AS raw_key"
                                                    , "CAST(value as String) AS raw_value",
                                                    " topic AS kafka_topic", "partition AS kafka_partition",
                                                    "offset AS kafka_offset", "timestamp AS kafka_timestamp")

In [ ]:
from pyspark.sql import functions as F

customer_bronze = (customer_converted
             .withColumn("is_tombstone", F.when(F.col("raw_value").isNull(), True).otherwise(False))
             .withColumn("ingested_at", F.current_timestamp() )
             )
customer_bronze.printSchema()

In [ ]:
from atlas.common.paths.loader import get_paths

paths = get_paths(settings)
bronze_customer_path = paths.bronze_path("customer/cdc/customers/notebook")
bronze_customer_checkpoint = paths.checkpoint_path("customer/cdc/customers/notebook")
print(bronze_customer_checkpoint)
print(bronze_customer_path)

In [ ]:
customer_bronze_to_write = customer_bronze.withColumn("ingested_date", F.to_date("ingested_at"))
query = (
    customer_bronze_to_write.writeStream.format("parquet")
    .outputMode("append")
    .option("checkpointLocation", bronze_customer_checkpoint)
    .partitionBy("ingested_date")
    .trigger(availableNow=True)
    .start(bronze_customer_path)
)

In [ ]:
customer_bronze_read = spark.read.format("parquet").load(bronze_customer_path)

In [ ]:
customer_bronze_read.printSchema()


In [ ]:
customer_bronze_read.count()

In [ ]:
customer_bronze_read.show(vertical=True)

In [ ]:
customer_bronze_read.filter(F.col("kafka_offset") == 14).select("raw_value").show(truncate=False)

In [ ]:
customer_bronze_read.filter(F.col("kafka_offset") == 10).show(truncate=False, vertical=True)